# Validation

Exercises top-level validation helpers, validation namespace exports, and validation errors.

Every `COVERAGE_CALLS` entry below is resolved against the current TorchLens API in this notebook.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
from torch import nn
from _shared import audit_touch_items, summarize_value  # noqa: E402


class AuditBufferModel(nn.Module):
    """Tiny model with a registered buffer for BufferLog coverage."""

    def __init__(self) -> None:
        """Initialize one linear layer and one buffer."""

        super().__init__()
        self.linear = nn.Linear(4, 2)
        self.register_buffer("offset", torch.ones(1, 4))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the buffer before the linear layer."""

        return self.linear(x + self.offset)


model = tiny_model(seed=0).train()
x = torch.randn(1, 4, requires_grad=True)
log = tl.trace(
    model,
    x,
    layers_to_save="all",
    save_grads=True,
    save_arg_values=True,
    save_rng_states=True,
    train_mode=True,
)
loss = log.layer_list[-1].out.sum()
log.log_backward(loss)
layer_pass = log.layer_list[1]
layer_log = log.layer_logs[layer_pass.layer_label_no_pass]
module_log = next(iter(log.modules))
module_pass = next(iter(module_log.ops.values()))
param_log = next(iter(log.params))
grad_fn_log = next(iter(log.grad_fns))
grad_fn_pass = next(iter(grad_fn_log.ops.values()))

buffer_log_model = AuditBufferModel().eval()
buffer_log_capture = tl.trace(buffer_log_model, torch.randn(1, 4), layers_to_save="all")
buffer_log = next(iter(buffer_log_capture.buffers))

clean, corrupt = make_clean_corrupt_pair(seed=0)
bundle = tl.bundle(
    [
        ("baseline", log),
        ("corrupt", tl.trace(tiny_model(seed=1).eval(), corrupt)),
    ],
    baseline="baseline",
)
AUDIT_CONTEXT = {
    "tl": tl,
    "Trace": log,
    "LayerLog": layer_log,
    "OpLog": layer_pass,
    "ModuleLog": module_log,
    "ModuleCallLog": module_pass,
    "ParamLog": param_log,
    "BufferLog": buffer_log,
    "GradFnLog": grad_fn_log,
    "GradFnCallLog": grad_fn_pass,
    "Bundle": bundle,
}
print(
    f"audit context: {len(log.layer_list)} layer ops, {len(log.modules)} modules, {len(log.grad_fns)} grad fns"
)

## Validation: concrete workflow

This cell demonstrates the user-facing workflow for this notebook before the manifest sweep.

In [ ]:
COVERAGE_CALLS = [
    "tl.errors.AppendBatchDependenceError",
    "tl.errors.AppendMismatchError",
    "tl.errors.AxisAmbiguityError",
    "tl.errors.BaselineUndeterminedError",
    "tl.errors.BatchNormTrainModeWarning",
    "tl.errors.BundleMemberError",
    "tl.errors.BundleNotFinalizedError",
    "tl.errors.BundleRelationshipError",
    "tl.errors.CaptureError",
    "tl.errors.CompatibilityError",
    "tl.errors.ConfigurationError",
    "tl.errors.ControlFlowDivergenceError",
    "tl.errors.ControlFlowDivergenceWarning",
    "tl.errors.DeadParentError",
    "tl.errors.DirectActivationWriteWarning",
    "tl.errors.DirectWriteIgnoredWarning",
    "tl.errors.DirectWriteInExecutableSaveError",
    "tl.errors.EngineDispatchError",
    "tl.errors.GraphShapeMismatchError",
    "tl.errors.HookSignatureError",
    "tl.errors.HookSiteCoverageError",
    "tl.errors.HookValueError",
    "tl.errors.InterventionAuditWarning",
    "tl.errors.InterventionError",
    "tl.errors.InterventionReadyConflictError",
    "tl.errors.LiveModeLabelError",
    "tl.errors.MetadataInvariantError",
    "tl.errors.ModelMismatchError",
    "tl.errors.MultiMatchWarning",
    "tl.errors.MutateInPlaceWarning",
    "tl.errors.NoParentError",
    "tl.errors.OpaqueCallableInExecutableSaveError",
    "tl.errors.PredicateError",
    "tl.errors.RecordContextFieldError",
    "tl.errors.RecorderStateError",
    "tl.errors.RecordingConfigError",
    "tl.errors.RecoveryError",
    "tl.errors.RecursiveTracingError",
    "tl.errors.ReplayPreconditionError",
    "tl.errors.Severity",
    "tl.errors.SiteAmbiguityError",
    "tl.errors.SiteResolutionError",
    "tl.errors.SpecMutationError",
    "tl.errors.SpecPortabilityError",
    "tl.errors.SpliceModuleDeviceError",
    "tl.errors.SpliceModuleDtypeError",
    "tl.errors.TorchLensError",
    "tl.errors.TorchLensIOError",
    "tl.errors.TorchLensInterventionError",
    "tl.errors.TorchLensInterventionWarning",
    "tl.errors.TorchLensPostfuncError",
    "tl.errors.TorchLensWarning",
    "tl.errors.TrainingModeConfigError",
    "tl.errors.UnsupportedTensorVariantError",
    "tl.errors.ValidationError",
    "tl.validate",
    "tl.validation.InterventionValidationReport",
    "tl.validation.MetadataInvariantError",
    "tl.validation.check_metadata_invariants",
    "tl.validation.check_spec_compat",
    "tl.validation.resolve_sites",
    "tl.validation.validate",
    "tl.validation.validate_backward_pass",
    "tl.validation.validate_batch_of_models_and_inputs",
    "tl.validation.validate_forward_pass",
    "tl.validation.validate_trace_saved_outs",
    "tl.validation.validate_saved_outs",
    "tl.validation.validate_tlspec",
]

for scope in ["quick", "full"]:
    try:
        print(
            scope,
            tl.validate(tiny_model(seed=6).eval(), torch.randn(1, 4), scope=scope, verbose=False),
        )
    except Exception as exc:
        print("# skipped validation", scope, type(exc).__name__, str(exc).splitlines()[0][:80])
print("validation exports", [name for name in dir(tl.validation) if not name.startswith("_")][:8])
print("errors", [name for name in dir(tl.errors) if "Error" in name][:8])

## Validation coverage cluster 1

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.errors.AppendBatchDependenceError",
    "tl.errors.AppendMismatchError",
    "tl.errors.AxisAmbiguityError",
    "tl.errors.BaselineUndeterminedError",
    "tl.errors.BatchNormTrainModeWarning",
    "tl.errors.BundleMemberError",
    "tl.errors.BundleNotFinalizedError",
    "tl.errors.BundleRelationshipError",
    "tl.errors.CaptureError",
    "tl.errors.CompatibilityError",
    "tl.errors.ConfigurationError",
    "tl.errors.ControlFlowDivergenceError",
    "tl.errors.ControlFlowDivergenceWarning",
    "tl.errors.DeadParentError",
    "tl.errors.DirectActivationWriteWarning",
    "tl.errors.DirectWriteIgnoredWarning",
    "tl.errors.DirectWriteInExecutableSaveError",
    "tl.errors.EngineDispatchError",
    "tl.errors.GraphShapeMismatchError",
    "tl.errors.HookSignatureError",
    "tl.errors.HookSiteCoverageError",
    "tl.errors.HookValueError",
    "tl.errors.InterventionAuditWarning",
    "tl.errors.InterventionError",
    "tl.errors.InterventionReadyConflictError",
    "tl.errors.LiveModeLabelError",
    "tl.errors.MetadataInvariantError",
    "tl.errors.ModelMismatchError",
    "tl.errors.MultiMatchWarning",
    "tl.errors.MutateInPlaceWarning",
    "tl.errors.NoParentError",
    "tl.errors.OpaqueCallableInExecutableSaveError",
]

audit_touch_items("Validation coverage cluster 1", ITEMS, AUDIT_CONTEXT)

## Validation coverage cluster 2

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.errors.PredicateError",
    "tl.errors.RecordContextFieldError",
    "tl.errors.RecorderStateError",
    "tl.errors.RecordingConfigError",
    "tl.errors.RecoveryError",
    "tl.errors.RecursiveTracingError",
    "tl.errors.ReplayPreconditionError",
    "tl.errors.Severity",
    "tl.errors.SiteAmbiguityError",
    "tl.errors.SiteResolutionError",
    "tl.errors.SpecMutationError",
    "tl.errors.SpecPortabilityError",
    "tl.errors.SpliceModuleDeviceError",
    "tl.errors.SpliceModuleDtypeError",
    "tl.errors.TorchLensError",
    "tl.errors.TorchLensIOError",
    "tl.errors.TorchLensInterventionError",
    "tl.errors.TorchLensInterventionWarning",
    "tl.errors.TorchLensPostfuncError",
    "tl.errors.TorchLensWarning",
    "tl.errors.TrainingModeConfigError",
    "tl.errors.UnsupportedTensorVariantError",
    "tl.errors.ValidationError",
    "tl.validate",
    "tl.validation.InterventionValidationReport",
    "tl.validation.MetadataInvariantError",
    "tl.validation.check_metadata_invariants",
    "tl.validation.check_spec_compat",
    "tl.validation.resolve_sites",
    "tl.validation.validate",
    "tl.validation.validate_backward_pass",
    "tl.validation.validate_batch_of_models_and_inputs",
]

audit_touch_items("Validation coverage cluster 2", ITEMS, AUDIT_CONTEXT)

## Validation coverage cluster 3

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.validation.validate_forward_pass",
    "tl.validation.validate_trace_saved_outs",
    "tl.validation.validate_saved_outs",
    "tl.validation.validate_tlspec",
]

audit_touch_items("Validation coverage cluster 3", ITEMS, AUDIT_CONTEXT)

In [ ]:
COVERAGE_CALLS = [
    "tl.errors.AppendBatchDependenceError",
    "tl.errors.AppendMismatchError",
    "tl.errors.AxisAmbiguityError",
    "tl.errors.BaselineUndeterminedError",
    "tl.errors.BatchNormTrainModeWarning",
    "tl.errors.BundleMemberError",
    "tl.errors.BundleNotFinalizedError",
    "tl.errors.BundleRelationshipError",
    "tl.errors.CaptureError",
    "tl.errors.CompatibilityError",
    "tl.errors.ConfigurationError",
    "tl.errors.ControlFlowDivergenceError",
    "tl.errors.ControlFlowDivergenceWarning",
    "tl.errors.DeadParentError",
    "tl.errors.DirectActivationWriteWarning",
    "tl.errors.DirectWriteIgnoredWarning",
    "tl.errors.DirectWriteInExecutableSaveError",
    "tl.errors.EngineDispatchError",
    "tl.errors.GraphShapeMismatchError",
    "tl.errors.HookSignatureError",
    "tl.errors.HookSiteCoverageError",
    "tl.errors.HookValueError",
    "tl.errors.InterventionAuditWarning",
    "tl.errors.InterventionError",
    "tl.errors.InterventionReadyConflictError",
    "tl.errors.LiveModeLabelError",
    "tl.errors.MetadataInvariantError",
    "tl.errors.ModelMismatchError",
    "tl.errors.MultiMatchWarning",
    "tl.errors.MutateInPlaceWarning",
    "tl.errors.NoParentError",
    "tl.errors.OpaqueCallableInExecutableSaveError",
    "tl.errors.PredicateError",
    "tl.errors.RecordContextFieldError",
    "tl.errors.RecorderStateError",
    "tl.errors.RecordingConfigError",
    "tl.errors.RecoveryError",
    "tl.errors.RecursiveTracingError",
    "tl.errors.ReplayPreconditionError",
    "tl.errors.Severity",
    "tl.errors.SiteAmbiguityError",
    "tl.errors.SiteResolutionError",
    "tl.errors.SpecMutationError",
    "tl.errors.SpecPortabilityError",
    "tl.errors.SpliceModuleDeviceError",
    "tl.errors.SpliceModuleDtypeError",
    "tl.errors.TorchLensError",
    "tl.errors.TorchLensIOError",
    "tl.errors.TorchLensInterventionError",
    "tl.errors.TorchLensInterventionWarning",
    "tl.errors.TorchLensPostfuncError",
    "tl.errors.TorchLensWarning",
    "tl.errors.TrainingModeConfigError",
    "tl.errors.UnsupportedTensorVariantError",
    "tl.errors.ValidationError",
    "tl.validate",
    "tl.validation.InterventionValidationReport",
    "tl.validation.MetadataInvariantError",
    "tl.validation.check_metadata_invariants",
    "tl.validation.check_spec_compat",
    "tl.validation.resolve_sites",
    "tl.validation.validate",
    "tl.validation.validate_backward_pass",
    "tl.validation.validate_batch_of_models_and_inputs",
    "tl.validation.validate_forward_pass",
    "tl.validation.validate_trace_saved_outs",
    "tl.validation.validate_saved_outs",
    "tl.validation.validate_tlspec",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")